*Notebook 05 of 12 in `notebooks/scripts_text/`. Parameters: `analysis/scripts_text/config.yaml`; decisions and run notes: `analysis/scripts_text/LOG.md`. The static page `site/scripts/index.html` is built from the same outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/scripts_text -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.scripts_text import s05_scatter as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 5 · Scattertext: which words does each group use comparatively more?

**What we do.** Each script is one document. For each group we add up how often every word is used and divide
by the group's total words, giving a **rate per 10,000 words**. We plot the unsuccessful group's rate (x)
against the successful group's rate (y) on log axes. The dashed diagonal is "same rate in both". Words near it
are shared language; words far above it are comparatively more frequent in successful scripts, words far
below it in unsuccessful ones. The distance from the diagonal is the score:

score = log10( (rate_successful + 0.1) / (rate_unsuccessful + 0.1) )

so a score of 0.3 means about twice as frequent. The 0.1 per 10,000 words is smoothing, so a word one group
never uses still gets a finite score.

**Why.** It is the course's two-corpus comparison (Scattertext, weeks 5-7). Using rates rather than counts
means the groups can differ in size and script length.

**Guarding against one script.** A word used 200 times in a single script can look distinctive by itself, so
a word must appear in at least **5 scripts** (document frequency, not total count). We also run 3 and 10, and
a "presence" mode that counts how many scripts use a word rather than how often.

**What it measures.** Which words each group uses comparatively more, in this corpus, under these
preprocessing choices.

**What it cannot tell us.** How a word is used (Step 8 reads the text), whether the difference is more than
chance (Step 6), whether it depends on which scripts are in the sample (Step 7), or anything about causes or
the quality of a script.

The primary run is definition (C), dialogue, names removed, minimum document frequency 5. All other runs are
exploratory.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

/Users/gyula/Documents/Projects/code/02805-social-graphs/.venv-course/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1111 11:22:45.776000 85547 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


**Figure.** Primary Scattertext: every dot is a word used by at least 5 scripts. x: rate in unsuccessful scripts, y: rate in successful scripts, both per 10,000 dialogue words on log scales. Words at least twice as frequent in one group are drawn as circles (successful) or triangles (unsuccessful); the labelled words are the most extreme on each side. Hover for counts. What to look for: whether the labelled words are plausible style or topic words or artefacts, which Step 8 checks in the text.

In [ ]:
step.figures(res)["primary"].show()

**Table.** The 25 most extreme words on each side of the primary run, with the number of scripts using each.

In [ ]:
display(step.tables(res)["top"])

,more frequent in successful,score (log10 ratio),scripts using it,more frequent in unsuccessful,score,scripts
0,je,0.91,19,jerry,-0.67,45
1,pas,0.89,18,eddie,-0.61,24
2,rocky,0.79,37,piper,-0.60,7
3,c'est,0.75,14,padre,-0.59,5
4,balboa,0.74,6,jake,-0.55,21
5,il,0.69,17,gem,-0.53,15
6,parker,0.69,22,sparky,-0.52,7
7,yah,0.68,34,jimmy,-0.51,57
8,chancellor,0.68,6,bloom,-0.50,19
9,vous,0.66,19,films,-0.49,26


**Figure.** Exploratory: the same plot for definition (A). With 67 unsuccessful scripts, rare words on the unsuccessful side are noisy.

In [ ]:
step.figures(res)["A"].show()

**Figure.** Exploratory: definition (B), which mixes genres and eras; expect genre words.

In [ ]:
step.figures(res)["B"].show()

**Figure.** Exploratory: the matched sample from Step 4 (same genre, decade and budget).

In [ ]:
step.figures(res)["matched"].show()

**Table.** Preprocessing variants of the primary run: how many terms clear the 2× threshold, and the Jaccard overlap of their top-50 lists with the primary run.

In [ ]:
display(step.tables(res)["variants"])

,variant,n_high,n_low,terms,words_high,words_low,above_threshold,above_high,above_low,jaccard_high_vs_primary,jaccard_low_vs_primary,top10_high,top10_low
0,primary,290,236,18719,2437561,2111709,210,124,86,1.000,1.000,"je, pas, rocky, c'est, balboa, il, parker, yah, chancellor, vous","jerry, eddie, piper, padre, jake, gem, sparky, jimmy, bloom, films"
1,names kept,290,236,19137,2500405,2163462,435,227,208,0.429,0.205,"rocky, jedi, caesar, je, pas, tre, barbie, elizabeth, creed, bella","kane, milton, susie, jennifer, lindsey, brennan, adam, fred, piper, craig"
2,names removed with NER too,290,236,17844,2394277,2075383,137,72,65,0.316,0.471,"rocky, ll, yah, chancellor, ce, tu, bender, tout, woody, de","rick, piper, abigail, steven, saudi, earl, ie, mop, gem, jerry"
3,stopwords removed,290,236,18431,1148609,995784,644,336,308,0.667,0.724,"je, pas, balboa, c'est, chancellor, rocky, parker, dans, il, sparrow","padre, eddie, piper, jerry, sparky, gem, sal, rico, ie, saudi"
4,lemmatised,290,236,14351,2387622,2068299,227,135,92,0.852,0.639,"je, pas, rocky, balboa, c'est, chancellor, parker, il, yah, vous","jerry, orchid, piper, padre, eddie, shaw, sparky, gem, mop, jake"


**Figure.** Jaccard overlap between each run's top-50 successful terms and the primary run's, across group definitions, layers, minimum document frequencies and count modes. Light = similar lists, dark = different.

In [ ]:
step.figures(res)["grid"].show()

**Table.** Mean overlap with the primary run by definition and layer.

In [ ]:
display(step.tables(res)["grid_summary"])

,definition,layer,jaccard_high_vs_primary,jaccard_low_vs_primary
0,A,action,0.02,0.02
1,A,dialogue,0.03,0.04
2,A,full,0.02,0.02
3,B,action,0.04,0.05
4,B,dialogue,0.22,0.16
5,B,full,0.11,0.06
6,C,action,0.05,0.05
7,C,dialogue,0.37,0.36
8,C,full,0.12,0.10
9,matched,action,0.00,0.02


**Table.** Within single genres and decades where both groups have at least 30 scripts: the ten most extreme words on each side.

In [ ]:
display(step.tables(res)["strata"])

,stratum,n_high,n_low,terms,words_high,words_low,above_threshold,above_high,above_low,top10_high,top10_low,jaccard_high_vs_primary,jaccard_low_vs_primary
0,genre: Adventure,57,40,5781,443319,287090,937,485,452,"jones, droids, chocolate, annie, obi, spider, republic, bo, harry, skywalker","joe, interview, spark, drug, drugs, nuclear, solar, bureau, texas, attorney",0.031,0.010
1,genre: Comedy,56,56,7461,554213,578327,609,235,374,"amy, harris, soccer, mason, harry, max, eagle, pilot, wise, mitchell","jerry, skate, jack, committee, rick, pudding, gates, script, august, lewis",0.010,0.064
2,genre: Drama,68,43,7075,621490,471552,718,384,334,"je, rocky, thou, tu, creed, du, grandma, de, elle, thee","mop, chairman, ray, players, football, jimmy, revolutionary, rosie, coach, olympic",0.124,0.075
3,genre: Horror,36,37,4307,203419,257853,796,331,465,"roger, michael, pattern, martin, lee, closet, peter, understanding, buzz, vacation","aye, el, blade, vampire, van, crew, cells, bio, hollow, military",0.010,0.010
4,decade: 1990s,60,49,7095,527181,491017,738,343,395,"andy, sheriff, buzz, thy, yah, reckon, chicago, aids, thee, thou","eddie, west, hunh, clark, tor, wrestling, peanut, johnson, lotta, ii",0.064,0.031
5,decade: 2000s,65,66,7405,553938,566715,708,297,411,"dragon, republic, master, princess, ram, council, donkey, shire, mum, harris","wi, saudi, comedian, handwriting, bloom, joe, dave, steven, yea, fbi",0.031,0.124
6,decade: 2010s,126,85,10338,1023565,700842,548,234,314,"amy, ll, houston, nigger, yah, songs, mason, bunny, thee, je","sheriff, mac, ski, olympic, mop, pause, ye, aliens, john, howard",0.111,0.111


**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

The primary comparison has 290 successful and 236 unsuccessful scripts (2,437,561 and 2,111,709 dialogue words) and 18,719 terms used by at least 5 scripts. 210 terms are at least twice as frequent in one group (124 on the successful side, 86 on the unsuccessful side). Whether that is more than chance is Step 6's question; on its own this number isn't a finding. Across the other 71 runs of the grid, the top-50 successful terms share a median Jaccard overlap of 0.02 with the primary run (unsuccessful side: 0.02); 1 would mean identical lists. Removing stopwords gives an overlap of 0.67, lemmatising 0.85, and keeping names 0.43. The explorers were searched for 117 eight-word runs of dialogue and contain 0.

Interactive Scattertext explorers for (C), (A), (B) and the matched sample are written to
`analysis/scripts_text/private/scattertext/`. They are built from word counts only, with no document text, and
are checked for dialogue before use, but stay out of git as the brief asks.